# Exploratory analysis - Identification of Brain Disease-Associated Genes using AI/ML Techniques

**Research/Educational Use Only.** This notebook demonstrates computational identification of gene-expression patterns
associated with Parkinson's disease in the public dataset **GSE7621**. It is not a diagnostic tool and the results
do not establish biological causality.

The notebook re-uses the same modules as the Streamlit app (`src/`). Run it from the `notebooks/` folder.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook"

from src import data_loader, preprocessing, eda, differential_expression as dea, ml_models, explainability as xai, visualization as viz
from src.utils import RANDOM_STATE, set_global_seed
set_global_seed(RANDOM_STATE)

## 1. Load and preprocess GSE7621
The first run downloads the GEO record (cached in `data/`).

In [ ]:
raw = data_loader.load_raw_dataset()
ds = preprocessing.build_processed_dataset(raw)
X, y = ds["X"], ds["y"]
X_desc = preprocessing.impute_median(X)
print("samples x genes:", X.shape)
print(y.map({0: "Control", 1: "Parkinson's"}).value_counts())
ds["metadata"][["title", "Class"]].head()

In [ ]:
# Documentation of the probe -> gene mapping and transformation decisions
{k: v for k, v in ds["report"].items() if k != "note"}

## 2. Quality control and exploratory analysis

In [ ]:
ds["qc"].round(3)

In [ ]:
coords, explained = eda.compute_pca(X_desc, n_components=3)
eda.pca_fig(coords, explained, y)

In [ ]:
eda.correlation_heatmap_fig(eda.correlation_matrix(X_desc, 1000), y)

In [ ]:
eda.top_variable_genes_fig(X_desc, 25)

## 3. Differential expression (Welch t-test + Benjamini-Hochberg FDR)

In [ ]:
de = dea.run_differential_expression(X_desc, y)
print("Genes with adjusted p <= 0.05:", int((de["Adjusted_P_Value"] <= 0.05).sum()))
de.head(15)

In [ ]:
viz.volcano_plot(de, max_adj_p=0.05, min_abs_log2fc=0.5)

## 4. Machine learning with nested feature selection
Median imputation, variance filtering, top-K gene selection and scaling are fitted **inside every training fold**
to prevent data leakage.

In [ ]:
results = ml_models.train_and_evaluate(X, y, de, model_choice="Compare All", n_features=50, n_splits=5, save=False)
results["cv_summary"][["Model", "Accuracy_Mean", "Accuracy_Std", "F1_Mean", "ROC_AUC_Mean", "ROC_AUC_Std", "Specificity_Mean"]].round(3)

In [ ]:
viz.model_comparison_fig(results["cv_summary"])

In [ ]:
viz.roc_curve_fig(results["oof"])

## 5. Which genes drive the models?
Coefficients, random-forest importance, permutation importance, SHAP and the project-specific consensus score.

In [ ]:
table, note = xai.importance_table_for("Consensus", results, de)
print(note)
table.head(20)

In [ ]:
viz.top_genes_bar(table, "Importance", 20, "Top 20 genes - consensus score")

In [ ]:
viz.expression_heatmap(X_desc, y, table["Gene"].head(25).tolist())

## 6. Limitations
Only 25 post-mortem samples, ~20,000 genes (p >> n), a single microarray platform, no independent validation cohort, and
unmodelled confounders (age, sex, post-mortem interval, cell-type composition). Genes listed here are *candidate associations*,
not proven biomarkers or causes.